In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd

from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DATASET_PATH = "../data/COVID_19_dataset/train"
DATASET_PATH_test = "../data/COVID_19_dataset/test"
DATASET_PATH_val = "../data/COVID_19_dataset/val"


In [ ]:
class_counts = {}

for cls in os.listdir(DATASET_PATH):

    cls_path = os.path.join(DATASET_PATH, cls)

    if os.path.isdir(cls_path):

        class_counts[cls] = len(
            os.listdir(cls_path)
        )

df = pd.DataFrame(
    class_counts.items(),
    columns=["Class", "Count"]
)

df

In [ ]:
class_counts = {}

for cls in os.listdir(DATASET_PATH_test):

    cls_path = os.path.join(DATASET_PATH_test, cls)

    if os.path.isdir(cls_path):

        class_counts[cls] = len(
            os.listdir(cls_path)
        )

df = pd.DataFrame(
    class_counts.items(),
    columns=["Class", "Count"]
)

df

In [ ]:
class_counts = {}

for cls in os.listdir(DATASET_PATH_val):

    cls_path = os.path.join(DATASET_PATH_val, cls)

    if os.path.isdir(cls_path):

        class_counts[cls] = len(
            os.listdir(cls_path)
        )

df = pd.DataFrame(
    class_counts.items(),
    columns=["Class", "Count"]
)

df

In [ ]:
def get_class_folders(path):

    return [
        d
        for d in os.listdir(path)
        if os.path.isdir(
            os.path.join(path, d)
        )
    ]

In [ ]:
classes = get_class_folders(DATASET_PATH)

fig, axes = plt.subplots(
    len(classes),
    3,
    figsize=(10,10)
)

for row, cls in enumerate(classes):

    cls_path = os.path.join(
        DATASET_PATH,
        cls
    )

    imgs = random.sample(
        os.listdir(cls_path),
        3
    )

    for col, img_name in enumerate(imgs):

        img_path = os.path.join(
            cls_path,
            img_name
        )

        img = Image.open(img_path)

        axes[row,col].imshow(
            img,
            cmap="gray"
        )

        axes[row,col].set_title(cls)

        axes[row,col].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
sizes = []

for cls in get_class_folders(DATASET_PATH):

    cls_path = os.path.join(
        DATASET_PATH,
        cls
    )

    for img_name in os.listdir(cls_path):

        img_path = os.path.join(
            cls_path,
            img_name
        )

        try:

            img = Image.open(img_path)

            sizes.append(
                img.size
            )

        except:

            pass

In [ ]:
print(
    "Unique Resolutions:",
    len(set(sizes))
)

print(
    pd.Series(sizes)
    .value_counts()
    .head(20)
)

In [ ]:
widths = [w for w,h in sizes]

plt.figure(figsize=(8,5))

plt.hist(
    widths,
    bins=30
)

plt.title(
    "Width Distribution"
)

plt.xlabel("Width")
plt.ylabel("Frequency")

plt.show()

heights = [h for w,h in sizes]

plt.hist(
    heights,
    bins=30
)

plt.title(
    "Height Distribution"
)

plt.xlabel("Height")
plt.ylabel("Frequency")

plt.show()

In [ ]:
modes = {}

for cls in get_class_folders(DATASET_PATH):

    cls_path = os.path.join(
        DATASET_PATH,
        cls
    )

    sample = os.listdir(cls_path)[0]

    img = Image.open(
        os.path.join(
            cls_path,
            sample
        )
    )

    modes[cls] = img.mode

modes

In [ ]:
bad_files = []

for cls in get_class_folders(DATASET_PATH):

    cls_path = os.path.join(
        DATASET_PATH,
        cls
    )

    for img_name in os.listdir(cls_path):

        img_path = os.path.join(
            cls_path,
            img_name
        )

        try:

            img = Image.open(img_path)

            img.verify()

        except:

            bad_files.append(
                img_path
            )

print(
    "Corrupted Images:",
    len(bad_files)
)

bad_files[:10]

In [ ]:
from torchvision import datasets
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(
        (224,224)
    ),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=transform
)

mean = 0.
std = 0.

for img,_ in dataset:

    mean += img.mean(
        [1,2]
    )

    std += img.std(
        [1,2]
    )

mean /= len(dataset)
std /= len(dataset)

print(
    "Mean:",
    mean
)

print(
    "Std:",
    std
)

## EDA Findings

- Dataset contains 3 balanced classes.
- Each class contains 1000 training images.
- Validation and test sets are also balanced.
- Images are grayscale chest X-rays.
- Resolution distribution varies across samples.
- No significant class imbalance exists.
- Images will be resized to 224x224.
- CLAHE preprocessing will be investigated.
- Standard CrossEntropyLoss will be used initially.